# 1

- The team's complete, two-block identical-prefix collision required an estimated $2^{63.1}$ calls to the SHA-1 compression function. Roughly 100-102 GPU-years of computation on NVIDIA K40/K80-class hardware.

- Because of variations in luck, code efficiency and the choice of GPUs/CPUs, the authors note that the same attack could plausibly cost anywhere between about $2^{62.3}$ and $2^{65.1}$ equivalent SHA-1 compression-function evaluations.

- The theoretical limit for the complexity of this attack is about $2^{61}$ SHA-1 compression calls. The practical attack is $\approx 2^{63.1}$, or about 2 orders of magnitude slower. Main reasons cited for this gap include:  
    - Loss of efficiency when running the complex attack algorithm on GPUs compared with the simplistic 'count SHA-1 calls' cost model.  
    - Overheads inherent to managing and scheduling a world-wide, large-scale distributed GPU computation.  
    - Extra complexity of engineering the second near-collision block beyond what the theoretical model assumed.

- A disturbance vector is a specially crafted, fully expanded message-difference pattern in which every '1' bit marks the start of a local collision. Selecting such a vector predetermines where message-word XOR differences occur and ensures they are compatible with SHA-1's linear message expansion, forming the backbone of the differential path used in the collision attack.

- A neutral bit is a message bit that, up to a given step $n$, can be flipped without (with high probability) violating any of the differential path conditions already satisfied. Because the diffusion in SHA-0/SHA-1's step function is relatively slow-bit differences propagate only gradually-many message bits affect the internal state only after several rounds, making it straightforward to locate numerous such neutral bits for the early steps.

- The second near-collision block of the attack employed 51 neutral bits, distributed over message words $m_{11}$ to $m_{15}$.

# 2

| Subject | Shared characteristics | Unique to SHA-1 | Unique to MD5 |
|---------|-----------------------|-----------------|---------------|
| **Basic design** | - Both are classic Merkle-Damgard hash functions.<br>- Work on 32-bit words with additions, rotations, and Boolean functions. | - Adds a message-word expansion (16 → 80 words) before processing each block.<br>- Uses 4 Boolean functions that change every 20 steps. | - No word expansion, the original 16 message words are reused in a permuted order.<br>- Uses 4 Boolean functions (F,G,H,I) fixed per 16-step round. |
| **Accepted inputs** | - Any bit-string/byte-string of arbitrary length. | - Treats input and internal words as big-endian. | - Treats input and internal words as little-endian. |
| **Maximum input size** | - 64-bit length field appended ⇒ both accept up to $2^{64} − 1$ bits. | n/a | n/a |
| **Padding operation** | - "1" bit, followed by "0" bits to reach 448 mod 512, then 64-bit length. | - Encodes length in big-endian.<br>- Uses message-word expansion after padding. | - Encodes length in little-endian.<br>- No expansion step. |
| **Internal buffer (chaining state)** | - State registers updated every compression step. | - Five 32-bit registers (A,B,C,D,E) → 160-bit state. | - Four 32-bit registers (A,B,C,D) → 128-bit state. |
| **Compression-function structure** | - Each step uses a non-linear Boolean function, modular addition, and rotation. | - 80 steps; message word schedule is data-dependent (XOR & rotate). | - 64 steps; message words used in a fixed, round-specific order. |
| **Number of steps / rounds** | - Multi-round structure with 4 logical phases. | - 80 steps $(4 \times 20)$. | - 64 steps $(4 \times 16)$. |
| **Message-block size** | - Each compresses 512-bit blocks. | n/a | n/a |
| **Generation of round constants** | - Fixed 32-bit constants mixed in every step. | - Four constants $(K_0...K_3)$, integer parts of $2^{30}\cdot \sqrt{j} \text{ where } j \text{-th prime in the sequence } \{2,3,5, 10\}$ | - Sixty-four constants $T_1...T_{64}$, integer parts of $2^{32} \cdot \|\sin (i)\| \text{ for } i = 1...64$ |
| **Digest size / output** | - Produce fixed-length message digests. | - 160-bit (20-byte) hash value. | - 128-bit (16-byte) hash value. |
| **Security status** | - Both are considered broken for collision resistance. | - First public collision (SHAttered) required $~2^{63}$ SHA-1 calls (2017); pre-image attacks still at brute-force $2^{160}$; somewhat stronger than MD5 but deprecated. | - Practical collisions since 2004; chosen-prefix collisions computable in hours on commodity hardware; essentially obsolete for all security uses. |

# 3

| **Subject** | **Shared characteristics** | **Unique to SHA-256** | **Unique to SHA-3** |
|---------|------------------|-------------------------|---------------------------|
| **Basic design** | - Approved by NIST.<br>- Use only simple bit-wise operations (rotations/shifts, XOR, AND, addition). | - Classical Merkle-Damgard construction with Davies-Meyer compression (an ARX design). | - Sponge construction that repeatedly applies a fixed permutation $ \text{Keccak-}f[1600] $. |
| **Accepted inputs** | - Accept a bit-string of arbitrary length. | - Treats message as a stream of big-endian 32-bit words. | - Endian-neutral bit stream; permutation is word-size agnostic. |
| **Maximum input size** | - In practice limited to below $2^{64}$ bits by FIPS documents. | - Limit is strict because a 64-bit length field is encoded in the padding. | - Padding has no length field; FIPS sets a procedural $2^{64}$-bit cap, but the sponge could absorb more theoretically. |
| **Padding rule** | - Start with a single ‘1’ bit, end on a block boundary. | - “$1\,0^{n}$” until the message length is $\equiv 448 \pmod{512}$; append 64-bit big-endian length (“MD-style” padding). | - Multi-rate rule “$10^{*}1$” followed by domain separator suffix 0x06 (or 0x1F for SHAKE). No length encoding. |
| **Internal state (buffer)** | - Maintain and update a fixed internal state throughout hashing. | - Eight 32-bit chaining words $(A,B,\dots,H)$ ⇒ $256$-bit state. | - Five-by-five array of 64-bit lanes ⇒ $5\times5\times64 = 1600$-bit state shared by both absorb & squeeze phases. |
| **Main step / operation** | - Combine message-dependent words, round constants and Boolean/arithmetic primitives each round. | - Message schedule $W[0\ldots63]$; each step uses $ \text{Ch},\text{Maj} $ and $\Sigma_0,\Sigma_1$ rotations; adds constant $K_i$. | - Each round executes five linear/non-linear mappings: $ \Theta,\;\rho,\;\pi,\;\chi,\;\iota $ over the full 1600-bit state. |
| **Number of rounds / steps** | - Multiple rounds give diffusion & non-linearity. | - 64 steps per 512-bit message block (organized as 4 logical rounds of 16 steps). | - 24 permutation rounds independent of message size. |
| **Block size processed at once** | - Both work with fixed-size chunks read from the message. | - $512$-bit blocks (the “rate” in MD terminology). | - Rate $r = 1088$ bits for SHA-3-256; capacity $c = 1600 - r = 512$ bits. (Other SHA-3 variants use different $r$.) |
| **Generation of constants** | - Rely on publicly known, non-secret constants. | - 64 constants $K_i = \lfloor 2^{32}\sqrt{\text{prime}_i}\rfloor$ for primes $2,3,\dots$. | - One round constant per permutation round, derived from an LFSR over $\mathbb{F}_2$ and injected in the $\iota$ step. |
| **Digest output** | - Produce a fixed-length hash value that can be truncated for shorter variants. | - 256-bit output; also defines SHA-224, SHA-512/256, etc. | - 256-bit output; also SHA-3-224/384/512 and extendable-output functions SHAKE128/256 (arbitrary length). |
| **Security status** | - No full practical breaks for either; reduced-round attacks exist. | - Susceptible to length-extension because of Merkle-Damgard; best collision attacks still require $>2^{112}$ work.<br>Widely trusted when used with HMAC or KDFs. | - Sponge design inherently resists length-extension and related-hash attacks; collision resistance $ = 2^{c/2} = 2^{256} $.<br>No known structural weaknesses; offers larger security margin. |

# 4

- $h$ be a Merkle-Damgard hash with (public) IV;  
- $K$ be a secret key, fixed for the receiver;  
- $\operatorname{MAC}(M)=h(M\;\|\;K)$ be the message-authentication code in question.

---
### (a)  Why $h(M\;\|\;K)$ is not vulnerable to the classical length-extension attack

- Length-extension relies on the attacker knowing the internal chaining state that exists after the secret key has been processed, so that they can continue hashing extra blocks without knowing the key.  

- When the key is prepended ($h(K\;\|\;M)$) the published tag equals the internal state afte* $K\!\!\parallel\!M$ is absorbed, hence the adversary can use that state as a 'starting IV' and append any padding + suffix $X$ to forge $\operatorname{MAC}(M\;\|\;\text{pad}\;\|\;X)=h(K\;\|\;M\;\|\;\text{pad}\;\|\;X)$.

- When the key is appended ($h(M\;\|\;K)$) the public tag is computed after the key itself has been injected. Knowing this final hash only reveals the state after $K$ has been processed; to extend the message the attacker would have to continue hashing without knowing $K$'s content, yet the next input block must start with (part of) $K$'s padding, which they cannot reconstruct correctly. Consequently, the adversary cannot compute $h(M\;\|\;K\;\|\;X)$ from $h(M\;\|\;K)$ alone, so the classic length-extension technique fails.

---

### (b)  A remaining vulnerability: exploiting an inner collision

Suppose the attacker can find two different messages $M_1\neq M_2$ such that  

$$
h(M_1)=h(M_2)\;.
$$

(this is a generic collision attack costing about $2^{n/2}$ operations
for an $n$-bit hash.)

Because Merkle-Damgard is iterative, having
$h(M_1)=h(M_2)$ means the chaining state after the last block of $M_1$ and $M_2$ is
identical.  Appending the same data afterward will therefore keep the digests equal:

$$
h(M_1\;\|\;K)=h(M_2\;\|\;K).
$$

Attack scenario  
- Offline, the adversary creates a collision pair $(M_1,M_2)$ with $h(M_1)=h(M_2)$.  
- They send $(M_1,\;T)$ to the legitimate party, where $T=\operatorname{MAC}(M_1)=h(M_1\;\|\;K)$, and obtains acceptance.  
- They now send $(M_2,\;T)$ as a new authenticated message. The receiver computes $h(M_2\;\|\;K)$, gets the same value $T$, and therefore accepts $M_2$ even though the sender never authorised it.

Thus, while appending the key thwarts length-extension, the construction remains
insecure against collision (or chosen-prefix) attacks.  Robust MAC designs such as
HMAC avoid both problems by hashing *key-derived* inner and outer paddings rather than
appending or prepending the raw key.

# 5

In [ ]:
import hashlib
from typing import List, Tuple

def int_to_bytes(value: int, length: int) -> bytes:
    """Convert a non-negative integer to a big-endian byte string of exactly *length* bytes."""
    return value.to_bytes(length, byteorder="big")

def find_three_byte_hits() -> Tuple[
        List[Tuple[bytes, bytes]],
        List[Tuple[bytes, bytes]],
        List[Tuple[bytes, bytes]]
    ]:
    """
    Returns three lists, one per hash algorithm.
    Each list contains tuples (msg, digest) for every 3-byte message whose
    digest starts with three zero bytes.
    """
    hits_sha1:   List[Tuple[bytes, bytes]] = []
    hits_sha512: List[Tuple[bytes, bytes]] = []
    hits_sha3_512: List[Tuple[bytes, bytes]] = []

    # There are exactly 2**24 = 16 777 216 candidates.
    for i in range(0x1000000):          # 0 … 0xFFFFFF inclusive
        msg = int_to_bytes(i, 3)       # 3-byte representation

        d1 = hashlib.sha1(msg).digest()
        if d1[:3] == b'\x00\x00\x00':
            hits_sha1.append((msg, d1))

        d2 = hashlib.sha512(msg).digest()
        if d2[:3] == b'\x00\x00\x00':
            hits_sha512.append((msg, d2))

        d3 = hashlib.sha3_512(msg).digest()
        if d3[:3] == b'\x00\x00\x00':
            hits_sha3_512.append((msg, d3))

    return hits_sha1, hits_sha512, hits_sha3_512

def print_hits(title: str, hits: List[Tuple[bytes, bytes]]) -> None:
    """Pretty-print the list of (msg, digest) pairs."""
    print(f"\n=== {title} ===")
    print(f"Number of 3-byte strings whose digest starts with 000000: {len(hits)}")
    if hits:
        print("msg (hex)".ljust(12), "→  digest (hex)")
        print("-" * 70)
        for msg, dg in hits:
            print(f"{msg.hex():12} → {dg.hex()}")

sha1_hits, sha512_hits, sha3_512_hits = find_three_byte_hits()

print_hits("SHA-1 (160-bit)", sha1_hits)
print_hits("SHA-2-512 (512-bit)", sha512_hits)
print_hits("SHA-3-512 (512-bit)", sha3_512_hits)


=== SHA‑1 (160‑bit) ===
Number of 3‑byte strings whose digest starts with 000000: 1
msg (hex)    →  digest (hex)
----------------------------------------------------------------------
03848c       → 000000b5bd75bab99ea944e18e50a862e3ff9090

=== SHA‑2‑512 (512‑bit) ===
Number of 3‑byte strings whose digest starts with 000000: 0

=== SHA‑3‑512 (512‑bit) ===
Number of 3‑byte strings whose digest starts with 000000: 1
msg (hex)    →  digest (hex)
----------------------------------------------------------------------
6c5e3b       → 0000000756f9676f106371421a0da820f252f8f9f65244d71b6d70252590d5d619dfe04a44d9230b8977f81f7d5c6ed07731518d8af09ca67555cfbd125ae0e2


In [ ]:
from multiprocessing import Pool, cpu_count


def _worker(args):
    length, start, stop, base_msg = args
    for i in range(start, stop):
        suffix = int_to_bytes(i, length) if length > 0 else b""
        if hashlib.sha3_512(base_msg + suffix).digest().startswith(b"\x00" * 4):
            return suffix
    return None


def find_shortest_suffix_parallel(base_msg: bytes, max_len: int = 8) -> Tuple[bytes, bytes]:
    ncpu = cpu_count()
    for length in range(max_len + 1):
        total = 256**length
        chunk = (total + ncpu - 1) // ncpu
        args = [
            (length, i * chunk, min(total, (i + 1) * chunk), base_msg)
            for i in range(ncpu)
        ]
        with Pool() as pool:
            for suffix in pool.imap_unordered(_worker, args):
                if suffix is not None:
                    digest = hashlib.sha3_512(base_msg + suffix).digest()
                    return suffix, digest
    raise RuntimeError("No solution found")


base = b"JHU695641"
print("\n=== Part (b) - shortest suffix for SHA3-512 ===")
print(f"Base message (hex): {base.hex()}")
suffix, digest = find_shortest_suffix_parallel(base_msg=base)
print(f"Shortest suffix (hex) : {suffix.hex()}")
print(f"Length of suffix      : {len(suffix)} byte(s)")
print(f"Resulting SHA3-512 digest (hex) : {digest.hex()}")
print(f"Digest starts with    : {digest[:4].hex()} (four zero bytes)")


=== Part (b) – shortest suffix for SHA3‑512 ===
Base message (hex): 4a4855363935363431
Shortest suffix (hex) : 976713f5
Length of suffix      : 4 byte(s)
Resulting SHA3‑512 digest (hex) : 00000000e50772b464b7246e26aae0908bfb08d27414b3d93c994435b162998ca8d781544e7a0d4553095062f95b18ecb5fb0c8e4a0e900f13e298eda7437c2b
Digest starts with    : 00000000 (four zero bytes)


In [ ]:
import cupy as cp
import hashlib
import math
import time
from typing import List, Tuple

# ----------------------------------------------------------------------
# CUDA C++ Device Code for SHA3-512 (Updated for Top-K Matches)
# ----------------------------------------------------------------------
cuda_source = r'''
typedef unsigned long long uint64_t;

// Keccak-1600 Round Constants
__device__ const uint64_t RC[24] = {
    0x0000000000000001ULL, 0x0000000000008082ULL, 0x800000000000808aULL,
    0x8000000080008000ULL, 0x000000000000808bULL, 0x0000000080000001ULL,
    0x8000000080008081ULL, 0x8000000000008009ULL, 0x000000000000008aULL,
    0x0000000000000088ULL, 0x0000000080008009ULL, 0x000000008000000aULL,
    0x000000008000808bULL, 0x800000000000008bULL, 0x8000000000008089ULL,
    0x8000000000008003ULL, 0x8000000000008002ULL, 0x8000000000000080ULL,
    0x000000000000800aULL, 0x800000008000000aULL, 0x8000000080008081ULL,
    0x8000000000008080ULL, 0x0000000080000001ULL, 0x8000000080008008ULL
};

extern "C" __global__
void find_shortest_suffixes(
    const unsigned char* base_msg,
    int base_len,
    int suffix_len,
    uint64_t start_idx,
    uint64_t num_elements,
    uint64_t* found_matches,
    int* match_count,
    int target_zeros,
    int max_results_per_chunk
) {
    uint64_t tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_elements) return;

    uint64_t current_val = start_idx + tid;

    // Build the message buffer
    unsigned char buffer[72]; 
    for(int i = 0; i < base_len; i++) {
        buffer[i] = base_msg[i];
    }
    
    // Convert current_val to big-endian bytes
    for(int i = 0; i < suffix_len; i++) {
        buffer[base_len + i] = (current_val >> (8 * (suffix_len - 1 - i))) & 0xFF;
    }
    int total_len = base_len + suffix_len;

    // Inline SHA3-512 (Keccak-f[1600])
    uint64_t A[25] = {0};
    unsigned char* S = (unsigned char*)A;

    // Absorb
    for(int i = 0; i < total_len; i++) S[i] = buffer[i];
    
    // SHA-3 domain separation & padding rule for SHA3-512
    S[total_len] ^= 0x06; 
    S[71] ^= 0x80;

    // 24 Rounds of Keccak Permutation
    for(int round = 0; round < 24; round++) {
        uint64_t C[5], D[5];
        for(int i=0; i<5; i++) C[i] = A[i] ^ A[i+5] ^ A[i+10] ^ A[i+15] ^ A[i+20];
        for(int i=0; i<5; i++) {
            uint64_t t = C[(i+1)%5];
            D[i] = C[(i+4)%5] ^ ((t << 1) | (t >> 63));
        }
        for(int i=0; i<25; i++) A[i] ^= D[i%5];

        uint64_t x = 1, y = 0, current = A[1];
        for(int i=0; i<24; i++) {
            int t_x = y;
            int t_y = (2*x + 3*y) % 5;
            uint64_t temp = A[t_x + 5*t_y];
            int r = ((i+1)*(i+2)/2) % 64;
            uint64_t rot_current = (r == 0) ? current : ((current << r) | (current >> (64-r)));
            A[t_x + 5*t_y] = rot_current;
            current = temp;
            x = t_x; y = t_y;
        }

        for(int j=0; j<25; j+=5) {
            uint64_t temp[5];
            for(int i=0; i<5; i++) temp[i] = A[j+i];
            for(int i=0; i<5; i++) A[j+i] ^= (~temp[(i+1)%5]) & temp[(i+2)%5];
        }
        A[0] ^= RC[round];
    }

    // Verify Condition
    bool match = true;
    for(int i = 0; i < target_zeros; i++) {
        if (S[i] != 0) {
            match = false;
            break;
        }
    }

    // Save ALL matches found up to the chunk's buffer limit safely
    if (match) {
        int idx = atomicAdd(match_count, 1);
        if (idx < max_results_per_chunk) {
            found_matches[idx] = current_val;
        }
    }
}
'''

# ----------------------------------------------------------------------
# Python Host Wrapping Code
# ----------------------------------------------------------------------
def hex_bytes(b: bytes) -> str:
    return b.hex()

def find_top_k_suffixes_gpu(base_msg: bytes, target_zero_bytes: int = 4, max_len: int = 8, k: int = 10) -> List[Tuple[bytes, bytes]]:
    module = cp.RawModule(code=cuda_source)
    kernel = module.get_function("find_shortest_suffixes")

    base_len = len(base_msg)
    d_base_msg = cp.array(bytearray(base_msg), dtype=cp.uint8)

    print(f"Executing search on GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

    found_results = []
    MAX_RESULTS_PER_CHUNK = 1024 # Safely generous buffer for a single chunk

    for length in range(max_len + 1):
        total_elements = 256 ** length
        
        threads_per_block = 256
        chunk_size = 100_000_000 
        
        # Chunks are sequential, meaning any matches in chunk N are strictly smaller than chunk N+1
        for start_idx in range(0, total_elements, chunk_size):
            elements_in_chunk = min(chunk_size, total_elements - start_idx)
            blocks_per_grid = math.ceil(elements_in_chunk / threads_per_block)

            # Initialize counters and arrays for this chunk
            d_found_matches = cp.zeros(MAX_RESULTS_PER_CHUNK, dtype=cp.uint64)
            d_match_count = cp.zeros(1, dtype=cp.int32)

            kernel((blocks_per_grid,), (threads_per_block,),
                   (d_base_msg,
                    cp.int32(base_len),
                    cp.int32(length),
                    cp.uint64(start_idx),
                    cp.uint64(elements_in_chunk),
                    d_found_matches,
                    d_match_count,
                    cp.int32(target_zero_bytes),
                    cp.int32(MAX_RESULTS_PER_CHUNK)))
            
            # Fetch matches back to the CPU
            match_count = int(d_match_count[0])
            if match_count > 0:
                num_to_read = min(match_count, MAX_RESULTS_PER_CHUNK)
                matches_in_chunk = d_found_matches[:num_to_read].get()
                
                # Because GPU threads finish unpredictably, sort them so we get strictly smallest values first
                matches_in_chunk.sort()
                
                for val in matches_in_chunk:
                    suffix = int(val).to_bytes(length, byteorder="big") if length > 0 else b''
                    
                    # Verify on CPU and get actual digest
                    digest = hashlib.sha3_512(base_msg + suffix).digest()
                    found_results.append((suffix, digest))
                    
                    # Stop returning once we hit the requested K amount
                    if len(found_results) == k:
                        return found_results

    return found_results

if __name__ == "__main__":
    base = b"JHU695641"
    num_to_find = 5
    print(f"\n=== GPU Accelerated - Finding top {num_to_find} shortest suffixes for SHA3-512 ===")
    print(f"Base message (hex): {hex_bytes(base)}")
    
    start_time = time.time()
    results = find_top_k_suffixes_gpu(base_msg=base, target_zero_bytes=4, max_len=8, k=num_to_find)
    end_time = time.time()

    print(f"\nSearch completed in: {end_time - start_time:.4f} seconds\n")
    
    for i, (suffix, digest) in enumerate(results, start=1):
        print(f"--- Match {i} ---")
        print(f"Suffix (hex)          : {hex_bytes(suffix)}")
        print(f"Length of suffix      : {len(suffix)} byte(s)")
        print(f"Resulting SHA3 digest : {hex_bytes(digest)}")


=== GPU Accelerated – Finding top 5 shortest suffixes for SHA3-512 ===
Base message (hex): 4a4855363935363431
Executing search on GPU: NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU

Search completed in: 22.6231 seconds

--- Match 1 ---
Suffix (hex)          : 7dfdac1c
Length of suffix      : 4 byte(s)
Resulting SHA3 digest : 0000000058f0622b069ba627705d559e325ea267a1ddd89ed22277d04c9067941a56806f43e19f27871175c7b20717c86ab3eccd9e7f3048a2350e081c29e9ec
--- Match 2 ---
Suffix (hex)          : 976713f5
Length of suffix      : 4 byte(s)
Resulting SHA3 digest : 00000000e50772b464b7246e26aae0908bfb08d27414b3d93c994435b162998ca8d781544e7a0d4553095062f95b18ecb5fb0c8e4a0e900f13e298eda7437c2b
--- Match 3 ---
Suffix (hex)          : fad000dd
Length of suffix      : 4 byte(s)
Resulting SHA3 digest : 000000000449ccee07bd56d64653d5af81d0013e26a5605ae20216a0ac9c2f9a55d1832972a54bc769fe5fe2a83cd96734ee071ef0517ae3887c1d41ef310889
--- Match 4 ---
Suffix (hex)          : 002e2656c8
Length of suff